# Attention Mechanism

Attention mechanism is a **communication mechanism** used to numerically compute the interaction between tokens.

For instance, let's have the following sentences: I have a dog. Its name is Max.

Between those two, which do you think should have a bigger value?

attention("its", "dog") compared with attention ("its", "I")

*its* is linked with the *dog* from the previous sentence. The role of the attention mechanism is to understand that given the context and to emphasize it for a better understanding.

That works for images as well. Suppose we have an image with a dog. After we divide the image into patches...some patches contain information about dog's ear, some about dog's eyes and some contains background info. We need a communication mechanism. 

To create that mechanism we need three matrices:
- **QUERY**: What I am looking for? $\implies$ which tokens are relevant
- **KEY**:   What information can I bring to the table? $\implies$ match with the queries to compute attention scores
- **VALUE**: How significant is that information? $\implies$ increase or decrease the attention scores

## Implement basic attention

In [90]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [91]:
B, P, E = 8, 65, 384
x = torch.rand((B, P, E))
x.shape

torch.Size([8, 65, 384])

In [92]:
head_size = 16

query = nn.Linear(E, head_size, bias=False)
key = nn.Linear(E, head_size, bias=False)
value = nn.Linear(E, head_size, bias=False)

q = query(x) # B x P x H
k = key(x) # B x P x H
v = value(x)

In [93]:
q.shape, k.shape, v.shape

(torch.Size([8, 65, 16]), torch.Size([8, 65, 16]), torch.Size([8, 65, 16]))

In [94]:
# ERROR
# q @ k -

In [95]:
qk = q @ k.transpose(-2, -1) # batch size is fixed... interchange only last two dimensions
qk.shape # B x P x P

torch.Size([8, 65, 65])

### Why to divide by head_size?

In [96]:
q_ex = torch.randn(10, 20)
k_ex = torch.randn(10, 20)

In [97]:
q_ex.var(), k_ex.var()

(tensor(1.0271), tensor(1.1111))

In [98]:
qk_ex = q_ex @ k_ex.transpose(-2, -1)
qk_ex.var()

tensor(20.7245)

The variance exploded... what does that mean tho?

In [99]:
x_ex = torch.tensor([0.1, -0.2, 0.3, -0.4, 0.2])
x_ex.var()

tensor(0.0850)

In [100]:
torch.softmax(x_ex, dim=0)

tensor([0.2140, 0.1585, 0.2613, 0.1298, 0.2365])

In [101]:
x_ex = 10 * x_ex
x_ex.var()

tensor(8.5000)

In [102]:
torch.softmax(x_ex, dim=0)

tensor([8.9575e-02, 4.4597e-03, 6.6187e-01, 6.0355e-04, 2.4349e-01])

It is important that the input of the softmax is fairly diffused because otherwise, softmax sharpen towards the maximum values and it gets way too picky during the attention mechanism: it will focus only towards 1 token, not towards multiple interactions. It is important at the beginning of the training when everything is randomly initialized, to force softmax to be as balanced as possible.

In [103]:
# BACK
qk_norm = qk * (head_size ** -0.5)

wei = F.softmax(qk_norm, dim=1) # per lines
wei.shape

out = wei @ v

In [104]:
out.shape

torch.Size([8, 65, 16])

In [105]:
out.var()

tensor(0.0632, grad_fn=<VarBackward0>)

In [106]:
class AttentionHead(nn.Module):
    def __init__(self, head_size, emb_size):
        super().__init__()
        
        self.head_size = head_size
        self.emb_size = emb_size
        
        self.query = nn.Linear(emb_size, head_size, bias=False)
        self.key = nn.Linear(emb_size, head_size, bias=False)
        self.value = nn.Linear(emb_size, head_size, bias=False)

    def forward(self, x):
        B, P, E = x.shape

        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        qk = q @ k.transpose(-1, -2) * (self.head_size ** -0.5)
        wei = F.softmax(qk, dim=-1)

        return wei @ v # B x P x H (where H = head_size)

In [120]:
num_heads = 16
head_size = E // num_heads
emb_size = E
heads = nn.ModuleList([AttentionHead(head_size, emb_size) for _ in range(num_heads)])

In [121]:
out = [head(x) for head in heads]
len(out)

16

In [113]:
out[0].shape

torch.Size([8, 65, 24])

In [115]:
torch.cat(out, dim=-1).shape

torch.Size([8, 65, 384])

In [130]:
class MultiHeadAttention(nn.Module):
    def __init__(self, head_size, num_heads, emb_size, dropout=0.2):
        super().__init__()

        assert num_heads * head_size == emb_size
        
        self.heads = nn.ModuleList([AttentionHead(head_size, emb_size) for _ in range(num_heads)])
        self.proj = nn.Linear(emb_size, emb_size)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.dropout(self.proj(out))

In [132]:
class Feedforward(nn.Module):
    def __init__(self, emb_size, dropout=0.2):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(emb_size, 4 * emb_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(4 * emb_size, emb_size),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

In [134]:
class Block(nn.Module):
    def __init__(self, num_heads, emb_size, dropout=0.2):
        super().__init__()

        assert emb_size % num_heads == 0
        
        head_size = emb_size // num_heads
        
        self.mah = MultiHeadAttention(head_size, num_heads, emb_size, dropout)
        self.ffwd = Feedforward(emb_size, dropout)
        self.ln1 = nn.LayerNorm(emb_size)
        self.ln2 = nn.LayerNorm(emb_size)

    def forward(self, x):
        x = x + self.mah(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [136]:
x.shape

torch.Size([8, 65, 384])

In [137]:
block = Block(
    num_heads=6,
    emb_size=384,
    dropout=0.2
)

output = block(x)

print("Input:", x.shape)
print("Output:", output.shape)

Input: torch.Size([8, 65, 384])
Output: torch.Size([8, 65, 384])


In [139]:
blocks = nn.Sequential(
    *[
        Block(num_heads=6, emb_size=384, dropout=0.2) for _ in range(8)
    ]
)

In [141]:
blocks(x).shape

torch.Size([8, 65, 384])